In [2]:
# Cell 0 — Setup (ALWAYS runs first)
import os
from google.colab import auth, userdata

# Authenticate
auth.authenticate_user()

# Clone repo if not already there
TOKEN = userdata.get('GITHUB_TOKEN')
if not os.path.exists("/content/wpa"):
    !git clone https://{TOKEN}@github.com/AbhirupDatta04/wpa.git /content/wpa

# CRITICAL — move into the repo folder
os.chdir("/content/wpa")
print("✅ Ready. Working in:", os.getcwd())
!ls

Cloning into '/content/wpa'...
remote: Enumerating objects: 31, done.
remote: Counting objects: 100% (31/31), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 31 (delta 6), reused 11 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (31/31), 18.29 KiB | 9.15 MiB/s, done.
Resolving deltas: 100% (6/6), done.
✅ Ready. Working in: /content/wpa
01_data_generation  03_processing  main_pipeline.ipynb	utils
02_ingestion	    04_analytics   README.md


In [3]:
# ============================================================
# utils/gcs_utils.ipynb — GCS helpers. %run after config.
# ============================================================
%run utils/config.ipynb

!pip install gcsfs google-cloud-storage -q

from google.cloud import storage
import pandas as pd, os

_client = storage.Client()

def upload_df(df, layer, name, fmt='parquet'):
    """Upload DataFrame to GCS layer. fmt = 'parquet' or 'csv'"""
    local = f'/tmp/{name}.{fmt}'
    if fmt == 'parquet':
        df.to_parquet(local, index=False)
    else:
        df.to_csv(local, index=False)
    gcs_path = PATHS[layer] + f'{name}.{fmt}'
    blob_path = gcs_path.replace(f'gs://{BUCKET_NAME}/', '')
    _client.bucket(BUCKET_NAME).blob(blob_path).upload_from_filename(local)
    print(f'  ✅ {name} → {gcs_path}')

def read_df(layer, name, fmt='parquet'):
    """Read DataFrame from GCS layer."""
    path = PATHS[layer] + f'{name}.{fmt}'
    opts = {'token': 'google_default'}
    if fmt == 'parquet':
        return pd.read_parquet(path, storage_options=opts)
    return pd.read_csv(path, storage_options=opts)

print('✅ GCS utils loaded')


✅ Config loaded
   Bucket: wealth-analytics-data-lake
   Layers: ['raw', 'bronze', 'silver', 'gold']
✅ GCS utils loaded
